In [1]:
import os, sys
import pandas as pd
import numpy as np
from pathlib import Path
from multiprocessing import get_context

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from greeks_calculator import compute_greeks

In [2]:
df = pd.read_csv("../data/vanilla_convertibles_data.csv")
print(f"Rows: {len(df):,}")
df.head()

Rows: 1,000,000


,S,r,q,bs_vol,credit_spread,redemption,coupon_rate,frequency,maturity_years,conversion_ratio,conversion_price,price_convertible
0,60.206715,0.071022,0.039404,0.735012,0.058539,100.0,0.097622,4,7.382009,3.903811,25.615993,251.871345
1,105.023875,0.027703,0.036579,0.636728,0.088371,100.0,0.033617,4,5.551895,3.345847,29.887795,351.393857
2,457.758451,0.085757,0.050711,0.296074,0.080561,100.0,0.015080,1,4.899940,6.646872,15.044670,3042.662012
3,356.330783,0.050178,0.010127,0.695026,0.063523,100.0,0.072563,4,7.476990,6.099752,16.394109,2173.529487
4,450.285185,0.059966,0.028056,0.191209,0.072044,100.0,0.044351,2,7.298957,5.683709,17.594146,2559.289745


In [3]:
def run_greeks_multi_cpu(
    df,
    n_procs=None,
    chunk_size=15_000,
    out_dir="greeks_output",
):
    out_dir = os.path.abspath(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    if n_procs is None:
        n_procs = max(1, os.cpu_count() - 2)

    splits = np.array_split(df, n_procs)

    jobs = []
    for wid, split_df in enumerate(splits):
        jobs.append((split_df.reset_index(drop=True), chunk_size, out_dir, wid))

    ctx = get_context("spawn")
    with ctx.Pool(processes=n_procs) as pool:
        pool.starmap(compute_greeks, jobs)

In [4]:
run_greeks_multi_cpu(df, n_procs=7, out_dir="greeks_output")

/opt/homebrew/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


KeyboardInterrupt: 

In [ ]:
out_dir = "greeks_output"
parts = []
for wid in range(7):
    chunk_id = 0
    while True:
        path = os.path.join(out_dir, f"w{wid}_greeks_chunk{chunk_id}.csv")
        if not os.path.exists(path):
            break
        parts.append(pd.read_csv(path))
        chunk_id += 1

greeks_df = pd.concat(parts, ignore_index=True)
print(f"Greeks rows: {len(greeks_df):,}")
greeks_df.describe()

In [ ]:
df["delta"] = greeks_df["delta"].values
df["gamma"] = greeks_df["gamma"].values
df["vega"]  = greeks_df["vega"].values

print(f"NaN count: delta={df['delta'].isna().sum()}, gamma={df['gamma'].isna().sum()}, vega={df['vega'].isna().sum()}")
df[["S", "bs_vol", "maturity_years", "price_convertible", "delta", "gamma", "vega"]].describe()

In [ ]:
df.to_csv("../data/vanilla_convertibles_data_greeks.csv", index=False)
print("Saved to data/vanilla_convertibles_data_greeks.csv")